# 00 Data Exploration — NFHS-5 Individual (Stage 1)

Exploratory analysis for the **individual-level** NFHS-5 women's recode (`NFHS5_Individual.csv`, 724,115 rows).

**Analytic sample decision (v5):** model the full 724,115 interviewed women. Barrier items (v467b–h) have zero missingness; some background/household modules are only populated for ~108K women and are encoded with a `missing` category during preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import BARRIER_SOURCE_COLS, load_stage1_data

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'NFHS5_Individual.csv'
print('Project root:', PROJECT_ROOT)
print('Raw path:', RAW_PATH)

In [ ]:
df = load_stage1_data()
print('Shape:', df.shape)
print('\nBarrier item missingness:')
for col in BARRIER_SOURCE_COLS:
    if col in df.columns:
        print(col, 'missing:', df[col].isnull().sum())
    else:
        print(col, 'NOT IN FILE')

print('\nBarrier "big problem" rates (individual items):')
for col in ['v467b', 'v467c', 'v467d', 'v467e', 'v467g', 'v467h']:
    if col in df.columns:
        rate = (df[col].astype(str).str.lower().str.strip() == 'big problem').mean()
        print(f'{col}: {rate*100:.1f}%')

from src.preprocessing.target_builder import build_targets
from src.preprocessing.clean import handle_missing

tmp = handle_missing(df.copy())
tmp = build_targets(tmp)

In [ ]:
from src.preprocessing.load_data import FEATURE_COLS

df = pd.read_csv(RAW_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print('Shape:', df.shape)
print('First 10 columns:', df.columns[:10].tolist())
print('\nDtypes (top 20):')
print(df.dtypes.head(20))

df.head(3)

In [ ]:
from src.preprocessing.load_data import FEATURE_COLS

feature_missing = pd.Series(
    {col: df[col].isna().mean() * 100 for col in FEATURE_COLS if col in df.columns}
).sort_values(ascending=False)

print('Feature missingness (% of rows):')
feature_missing.head(10)

In [ ]:
# Individual-level distributions (replaces old district-level indicator histograms)
plot_specs = [
    ('v012', 'Age (years)', 'hist'),
    ('v106', 'Education (v106)', 'count'),
    ('v467b', 'Barrier: getting money (v467b)', 'count'),
    ('v467g', 'Barrier: distance to facility (v467g)', 'count'),
]

available = [(col, title, kind) for col, title, kind in plot_specs if col in df.columns]
if not available:
    print('No plot columns found in dataframe.')
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, (col, title, kind) in zip(axes, available):
        if kind == 'hist':
            sns.histplot(pd.to_numeric(df[col], errors='coerce').dropna(), kde=True, ax=ax)
        else:
            order = df[col].astype(str).str.strip().value_counts().index
            sns.countplot(y=df[col].astype(str).str.strip(), order=order, ax=ax)
        ax.set_title(title)

    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation among numeric model features (age + media exposure items)
numeric_feature_cols = [
    c for c in FEATURE_COLS
    if c in df.columns and pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors='coerce'))
]

numeric_df = df[numeric_feature_cols].apply(pd.to_numeric, errors='coerce')

if numeric_df.shape[1] < 2:
    print('Not enough numeric feature columns for a heatmap.')
else:
    plt.figure(figsize=(8, 6))
    sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation heatmap — numeric Stage 1 features')
    plt.tight_layout()
    plt.show()